## bing crawl with actual soruces saved from bing redirects
- sources are on paragraph level not like google which is one whole generation level
- all links are redirects which need extra step to get actual links and make sure they are real/accesable
- no pixel level coordinates given for bing engine

In [ ]:
# packages
!pip install google-search-results

In [ ]:
# API keys

serpapit_keys = 

key_in_use = serpapit_keys[2][1]  # change index to switch keys
key_in_use


In [2]:
# bing original source link helper

import re
import base64
import html
import requests
from urllib.parse import urlparse, parse_qs, unquote

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0"}


def _is_probably_url(s: str) -> bool:
    """Basic sanity check to ensure we only return real URLs."""
    if not s:
        return False
    s = s.strip()
    p = urlparse(s)
    return p.scheme in ("http", "https", "ftp") and bool(p.netloc)


def _b64_urlsafe_decode(s: str) -> str:
    s = s.strip().replace(" ", "+")
    pad = (-len(s)) % 4
    return base64.urlsafe_b64decode(s + "=" * pad).decode("utf-8", "ignore")


def _b64_standard_decode(s: str) -> str:
    s = s.strip().replace(" ", "+")
    pad = (-len(s)) % 4
    return base64.b64decode(s + "=" * pad).decode("utf-8", "ignore")


def _extract_b64_substring(raw: str) -> str:
    """
    Bing often puts a base64 URL that starts with 'aHR0' (='http').
    We only take the contiguous base64-ish run starting there, to avoid
    decoding random trailing characters.
    """
    i = raw.find("aHR0")
    if i < 0:
        return ""

    # Only allow base64-ish chars after aHR0
    m = re.match(r"[A-Za-z0-9+/=_-]+", raw[i:])
    if not m:
        return ""
    return m.group(0)


def _maybe_decode_u(val: str) -> str:
    """Decode Bing u= value. Handles prefixes and double-encoding."""
    # First decode URL-encoding
    raw = unquote(val)

    # 1) Already a normal URL (e.g., https%3A%2F%2Fexample.com...)
    if _is_probably_url(raw):
        return raw.strip()

    # 2) Try to extract a base64 chunk starting with 'aHR0'
    raw_b64 = _extract_b64_substring(raw)
    if not raw_b64:
        return ""

    def _decode_attempt(decoder, s: str) -> str:
        try:
            return decoder(s)
        except Exception:
            return ""

    # Try urlsafe, then standard base64
    decoded = _decode_attempt(_b64_urlsafe_decode, raw_b64)
    if not decoded:
        decoded = _decode_attempt(_b64_standard_decode, raw_b64)

    if decoded:
        # Some Bing payloads: decoded is still URL-encoded (https%3A%2F%2F...)
        decoded_unquoted = unquote(decoded)

        for candidate in (decoded_unquoted, decoded):
            candidate = candidate.strip()
            if _is_probably_url(candidate):
                return candidate

    # If we got here, we didn't find a safe URL
    return ""


def _follow_html_redirects(url: str) -> str:
    """Handles cases where Bing serves a 200 + meta refresh / JS redirect."""
    try:
        r = requests.get(url, headers=UA, timeout=15, allow_redirects=True)

        # If real 3xx happened, requests already followed it
        final = r.url.strip()

        # Filter out Bing tracking endpoints
        if _is_probably_url(final) and not any(
            final.startswith(prefix)
            for prefix in (
                "https://www.bing.com/ck/",
                "https://bing.com/ck/",
                "https://www.bing.com/aclk",
                "https://bing.com/aclk",
            )
        ):
            return final

        # Parse <meta http-equiv="refresh" ... url=...>
        m = re.search(
            r'http-equiv=["\']refresh["\'][^>]*url=([^"\'> ]+)',
            r.text,
            re.I,
        )
        if m:
            candidate = html.unescape(unquote(m.group(1))).strip()
            if _is_probably_url(candidate):
                return candidate

        # Simple window.location = "..."
        m = re.search(
            r'window\.location(?:\.href)?\s*=\s*[\'"]([^\'"]+)[\'"]',
            r.text,
            re.I,
        )
        if m:
            candidate = html.unescape(unquote(m.group(1))).strip()
            if _is_probably_url(candidate):
                return candidate

    except requests.RequestException:
        pass
    return ""


def resolve_bing_redirect(raw_url: str) -> str:
    """
    Resolve Bing / MSN click-tracking URLs. For non-Bing URLs, just
    return the original string (no decoding / HTML fetching).
    """
    raw_url = raw_url.strip()
    if not raw_url:
        return raw_url

    p = urlparse(raw_url)
    qs = parse_qs(p.query)

    # 0) If it's already a normal non-Bing URL, don't touch it
    if not p.netloc.endswith(("bing.com", "msn.com", "r.msn.com", "go.msn.com")):
        return raw_url

    # 1) Bing click trackers: organic (/ck/...) and ads (/aclk)
    if p.netloc.endswith("bing.com") and (
        p.path.startswith("/ck/") or p.path.startswith("/aclk")
    ):
        if "u" in qs and qs["u"]:
            dest = _maybe_decode_u(qs["u"][0])
            if _is_probably_url(dest):
                return dest

        # Fallback to other common params if present
        for key in ("url", "r"):
            if key in qs and qs[key]:
                candidate = unquote(qs[key][0]).strip()
                if _is_probably_url(candidate):
                    return candidate

    # 2) MSN forwarders
    if p.netloc.endswith(("r.msn.com", "go.msn.com", "msn.com")):
        for key in ("r", "u", "url"):
            if key in qs and qs[key]:
                candidate = unquote(qs[key][0]).strip()
                if _is_probably_url(candidate):
                    return candidate

    # 3) Fallback: fetch and parse HTML redirects (only for Bing/MSN)
    follow = _follow_html_redirects(raw_url)
    if _is_probably_url(follow):
        return follow

    # 4) Give up gracefully: return the original (Bing) URL
    return raw_url

## google crawl

In [3]:
from serpapi import GoogleSearch
import json
import requests


def fetch_google_ai_data(token, api_key):
    params = {
    "engine": 'google_ai_overview',
    "page_token": token,
    "api_key": api_key,
    }

    search = GoogleSearch(params)
    results = search.get_dict()
    if "ai_overview" in results:
      return results["ai_overview"]     
    else:
      print("\033[1;31m Error in PAGE TOKEN -- No AI overview found\033[0m")
      return None



def fetch_google_data(q, api_key, location=None):
  params = {
    "engine": 'google',
    "q": q,
    "api_key": api_key,
    "location": location,
      "device":'mobile',
      "no_cache": True
  }

  search = GoogleSearch(params)
  results = search.get_dict()

  if "ai_overview" in results:
    if "page_token" in results["ai_overview"]:
      ai_o_serpapi_link = results["ai_overview"]["page_token"]
      aio=fetch_google_ai_data(ai_o_serpapi_link,key_in_use)
      return results,aio
    else:
      return results,results["ai_overview"]     
  else:
      #print(f"\033[1;31mNo AI overview found\033[0m : {q}")
      return results,None


In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm

# for query in queries from the data/combined_sampled_queries.csv file, do the fetch_google_data function and save the results like this: data is saved as combined_sampled_queries_{query_index}_google_search.json and ai_overview is saved as combined_sampled_queries_{query_index}_google_ai_overview.json

# Load queries from the CSV file
queries_df = pd.read_csv("google_trends_AIO_data.csv")
output_path = "AIO_data_collection/google/trends"
no_AIO_file="AIO_data_collection/google/trends/noAIO.jsonl"
# Ensure the output directory exists
os.makedirs(output_path, exist_ok=True)
os.makedirs(os.path.dirname(no_AIO_file), exist_ok=True)
# Sort the dataframe by ID in ascending order
queries_df = queries_df.sort_values(by="ID")
#queries_df=queries_df[132:]
queries_to_run = queries_df[queries_df['ID'] >= 0]
total_queries = len(queries_to_run)

processed_queries = 0
skipped_queries = 0
crawl_start_time = time.time()

# Iterate through the queries and fetch data
with tqdm(total=total_queries,desc="Starting",unit='query') as pbar:
    for index, row in queries_to_run.iterrows():
        search_file = f"{output_path}/stratified_benchmark_test_{row['ID']}_google_search.json"
        ai_file = f"{output_path}/stratified_benchmark_test_{row['ID']}_google_ai_overview.json"
        if os.path.exists(search_file) and os.path.exists(ai_file):
            skipped_queries += 1
            pbar.update(1)
            print(f"Skipping query index {row['ID']}: results already downloaded")
        else:
            #print(f"Processing query index {row['ID']}: {row['query']}")
            try:
                query = row['query']
                query_index = row['ID']
                data, ai_overview = [], []
                # Fetch Google data
                data, ai_overview = fetch_google_data(query, key_in_use, location="A city in NE of US, STATE")
                #data, ai_overview = fetch_google_data(query, key_in_use)

                # Save the results
                with open(search_file, "w") as f:
                    json.dump(data, f, indent=4)
                    #print(f"Saved Google search results for query index {query_index}")
                if ai_overview is not None:
                    with open(ai_file, "w") as f:
                        json.dump(ai_overview, f, indent=4)
                        #print(f"Saved Google AI overview for query index {query_index}\n")
                else:
                    with open(no_AIO_file, "a") as f:
                        f.write(json.dumps(query) + "\n")
            
            
            
            except Exception as e:
                tqdm.write(f"Error on ID {row['ID']}: {e}")
            
            pbar.update(1)
    #elapsed = time.time() - crawl_start_time
    #completed = processed_queries + skipped_queries
    #remaining = total_queries - completed
    #avg_time_per_query = elapsed / processed_queries if processed_queries else 0
    #est_remaining = avg_time_per_query * remaining
    #print(
    #    f"Progress: {completed}/{total_queries} IDs handled ({skipped_queries} skipped). "
    #    f"Remaining IDs: {remaining}. Estimated time left: {est_remaining:.1f}s."
    #)

Starting:   1%|          | 10/1726 [00:42<2:01:41,  4.26s/query]

In [34]:
ids=[1438]

In [32]:
len(ids)

3

In [36]:
## second round for failed page tokens

queries_df = pd.read_csv("AIO_Benchmark_Dataset.csv")
output_path = "AIO_data_collection/google/A city in NE of US"

queries_to_run=queries_df[queries_df.ID.isin(ids)]

total_queries = len(queries_to_run)


crawl_start_time = time.time()

# Iterate through the queries and fetch data
with tqdm(total=total_queries,desc="Starting",unit='query') as pbar:
    for index, row in queries_to_run.iterrows():
        search_file = f"{output_path}/stratified_benchmark_test_{row['ID']}_google_search.json"
        ai_file = f"{output_path}/stratified_benchmark_test_{row['ID']}_google_ai_overview.json"
        if os.path.exists(search_file) and os.path.exists(ai_file):
            pbar.update(1)
            print(f"Skipping query index {row['ID']}: results already downloaded")
        else:
            #print(f"Processing query index {row['ID']}: {row['query']}")
            try:
                query = row['query']
                print(query)
                query_index = row['ID']
                data, ai_overview = [], []
                # Fetch Google data
                data, ai_overview = fetch_google_data(query, key_in_use, location="A city in NE of US, STATE")
                #data, ai_overview = fetch_google_data(query, key_in_use)

                # Save the results
                with open(search_file, "w") as f:
                    json.dump(data, f, indent=4)
                    #print(f"Saved Google search results for query index {query_index}")
                if ai_overview is not None:
                    with open(ai_file, "w") as f:
                        json.dump(ai_overview, f, indent=4)
                        #print(f"Saved Google AI overview for query index {query_index}\n")
                else:
                    with open(no_AIO_file, "a") as f:
                        f.write(json.dumps(query) + "\n")
            
            
            
            except Exception as e:
                tqdm.write(f"Error on ID {row['ID']}: {e}")
            
            pbar.update(1)


Starting:   0%|          | 0/1 [00:00<?, ?query/s]

What are the best IWB holsters for Glock 19?


Starting: 100%|██████████| 1/1 [00:11<00:00, 11.66s/query]


In [30]:
# bing function 
from serpapi import GoogleSearch
import json
import requests

def fetch_bing_data(q, api_key):
  params = {
    "engine": "bing",
    "q": q,
    "api_key": api_key
  }

  search = GoogleSearch(params)
  results = search.get_dict()

  if "answer_box" in results:
      return results,results["answer_box"]     
  else:
      print(f"\033[1;31mNo answer_box found\033[0m: {q}")
      return results,None


In [7]:
# run
import pandas as pd
import json
import os
import time
from tqdm.notebook import tqdm

# for query in queries from the data/combined_sampled_queries.csv file, do the fetch_google_data function and save the results like this: data is saved as combined_sampled_queries_{query_index}_google_search.json and ai_overview is saved as combined_sampled_queries_{query_index}_google_ai_overview.json

# Load queries from the CSV file
queries_df = pd.read_csv("AIO_Benchmark_Dataset.csv")
output_path = "AIO_data_collection/bing/no_location_forced"

no_AIO_file="AIO_data_collection/bing/no_location_forced/noAIO.jsonl"
# Ensure the output directory exists
os.makedirs(output_path, exist_ok=True)
os.makedirs(os.path.dirname(no_AIO_file), exist_ok=True)

# Sort the dataframe by ID in ascending order
queries_df = queries_df.sort_values(by="ID")
queries_to_run = queries_df[queries_df['ID'] >= 1]
total_queries = len(queries_to_run)

processed_queries = 0
skipped_queries = 0
crawl_start_time = time.time()

# Iterate through the queries and fetch data
with tqdm(total=total_queries,desc="Starting",unit='query') as pbar:
    for index, row in queries_to_run.iterrows():
        search_file = f"{output_path}/stratified_benchmark_test_{row['ID']}_bing_search.json"
        ai_file = f"{output_path}/stratified_benchmark_test_{row['ID']}_bing_ai_overview.json"
        if os.path.exists(search_file) and os.path.exists(ai_file):
            skipped_queries += 1
            pbar.update(1)
            print(f"Skipping query index {row['ID']}: results already downloaded")
        else:
            #print(f"Processing query index {row['ID']}: {row['query']}")
            try:
                query = row['query']
                query_index = row['ID']
                data, ai_overview = [], []

                # Fetch Bing data
                data, ai_overview = fetch_bing_data(query, key_in_use)

        #data, ai_overview = fetch_google_data(query, key_in_use)

                # Save the results
                with open(search_file, "w") as f:
                    json.dump(data, f, indent=4)
                    #print(f"Saved Google search results for query index {query_index}")
                if ai_overview is not None:
                    with open(ai_file, "w") as f:
                        json.dump(ai_overview, f, indent=4)
                        #print(f"Saved Google AI overview for query index {query_index}\n")
                        
                else:
                    with open(no_AIO_file, "a") as f:
                        f.write(json.dumps(query) + "\n")
                processed_queries += 1
            except Exception as e:
                tqdm.write(f"Error on ID {row['ID']}: {e}")
            
            pbar.update(1)
   # elapsed = time.time() - crawl_start_time
   # completed = processed_queries + skipped_queries
   # remaining = total_queries - completed
   # avg_time_per_query = elapsed / processed_queries if processed_queries else 0
   # est_remaining = avg_time_per_query * remaining
   # print(
   #     f"Progress: {completed}/{total_queries} IDs handled ({skipped_queries} skipped). "
   #     f"Remaining IDs: {remaining}. Estimated time left: {est_remaining:.1f}s."
   # )

Starting:   0%|          | 0/11500 [00:00<?, ?query/s]

No answer_box found: holidaytraditions
No answer_box found: youtube camera for vlogging
No answer_box found: picasso tiles
No answer_box found: electronic bug detector
No answer_box found: boxers for men
No answer_box found: physionatural
No answer_box found: tablet xp pen
No answer_box found: large grow bag
No answer_box found: palmtalkhome
No answer_box found: computer repair tool kit
No answer_box found: emergency solar crank radio
No answer_box found: xp pen
No answer_box found: portable bluetooth speakers with lights
No answer_box found: 8th generation ipad case with keyboard
No answer_box found: pull patch
No answer_box found: toddler tablets for 2 year old with wifi
No answer_box found: laser level
No answer_box found: heated gloves
No answer_box found: brooklyn botany
No answer_box found: smooth viking
No answer_box found: womens columbian jeans
No answer_box found: vlogging camera
No answer_box found: reusable cotton produce bags
No answer_box found: kastking baitcasting reels


KeyboardInterrupt



In [ ]:
# improve code for better efficiency, now it is super slow: from pathlib import Path
import json


# Define the directories
directories = [
    # "data/stratified_benchmark_test/bing_search_results_location_forced_A city in NE of US_New_Jersey",
    "data/stratified_benchmark_test/bing_search_results_no_forced_location"
]

# Function to resolve links in JSON files
def process_json_files(directory):
    json_files = list(Path(directory).glob("*.json"))
    for json_file in json_files:
        try:
            # Load the JSON content
            with open(json_file, "r") as f:
                content = json.load(f)
            
            # Define the helper function to resolve links
            def search_key_replace_redirect(data, key):
                if isinstance(data, dict):
                    for k, v in data.items():
                        if k == key:
                            data[k] = resolve_bing_redirect(v)
                        else:
                            search_key_replace_redirect(v, key)
                elif isinstance(data, list):
                    for item in data:
                        search_key_replace_redirect(item, key)
            
            # Resolve links in the JSON content
            search_key_replace_redirect(content, "tracking_link")
            search_key_replace_redirect(content, "link")
            
            # Save the updated JSON content back to the file
            with open(json_file, "w") as f:
                json.dump(content, f, indent=4)
            print(f"Processed file: {json_file}")
        except Exception as e:
            print(f"Failed to process file {json_file}: {e}")

directories.sort()

# Process each directory
for directory in directories:
    process_json_files(directory)

Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/11387_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/6021_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/9256_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/9152_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/7485_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/1103_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/3409_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/11222_bing_search.json
Processed file: data/stratified_benchmark_test/bing_search_results_no_forced_location/71_bing_search.json
Processed file: data/stratif